In [ ]:
# CELL 1
# Install dependencies

!pip install transformers
!pip install sentencepiece
!pip install torch
!pip install newspaper3k
!pip install lxml_html_clean
!pip install rouge_score

In [ ]:
# CELL 2
# Import libraries

import re
import torch

from newspaper import Article
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
# CELL 3
# Load model and tokenizer
# Model: mT5 fine-tuned on XL-Sum (includes Indonesian news articles)

MODEL_NAME = 'csebuetnlp/mT5_multilingual_XLSum'

print(f'Loading model: {MODEL_NAME}')
print('This may take a few minutes on first run (model ~1.2GB)...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

print(f'\nModel loaded. Running on: {device}')

In [ ]:
# CELL 4
# Input article URL

url = '-'

In [ ]:
# CELL 5
# Download and parse article

article = Article(url, language='id')
article.download()
article.parse()

title = article.title
text  = article.text

print('TITLE:')
print(title)
print('\nARTICLE (first 1500 chars):')
print(text[:1500])
print(f'\nTotal characters: {len(text)}')
print(f'Total words     : {len(text.split())}')

In [ ]:
# CELL 6
# Text cleaning function
# Removes noisy artifacts common in scraped Indonesian news articles

def clean_text(text: str) -> str:

    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Remove lines that are likely captions or image labels (very short, all caps)
    lines = text.splitlines()
    lines = [
        line for line in lines
        if not (len(line.strip()) < 60 and line.strip().isupper())
    ]
    text = '\n'.join(lines)

    # Collapse extra whitespace
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r' {2,}', ' ', text)

    return text.strip()

In [ ]:
# CELL 7
# Chunk splitter
#
# mT5-XLSum has a 512 token input limit.
# For long articles we split into overlapping word-based chunks,
# summarize each, then concatenate the partial summaries for
# a final summarization pass (hierarchical summarization).

def split_into_chunks(
    text: str,
    max_words: int = 400,
    overlap_words: int = 50
) -> list:

    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = min(start + max_words, len(words))
        chunk = ' '.join(words[start:end])
        chunks.append(chunk)

        if end == len(words):
            break

        start += max_words - overlap_words

    return chunks

In [ ]:
# CELL 8
WHITESPACE_HANDLER = lambda text: re.sub(r'\n+', ' . ', text.strip())

def summarize_chunk(
    text: str,
    # Adjust these parameters to control the length of the generated summaries
    max_new_tokens: int = 300, # Increased maximum tokens
    min_new_tokens: int = 100,  # Increased minimum tokens
    # More beams = model explores more candidate sequences
    num_beams=8,
    length_penalty=1.2,   # > 1.0 encourages longer, more complete summaries
    no_repeat_ngram_size: int = 3
) -> str:

    prepared = WHITESPACE_HANDLER(text)

    inputs = tokenizer(
        prepared,
        return_tensors='pt',
        padding='max_length',
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        output_ids = model.generate(
            inputs['input_ids'],
            attention_mask=inputs['attention_mask'],
            max_new_tokens=max_new_tokens,
            min_new_tokens=min_new_tokens,
            num_beams=num_beams,
            length_penalty=length_penalty,
            no_repeat_ngram_size=no_repeat_ngram_size,
            repetition_penalty=1.8, # Slightly reduced repetition penalty
            early_stopping=True
        )

    summary = tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True
    )

    return summary

In [ ]:
# CELL 9
# Hierarchical summarizer
#
# Short articles  (<= 400 words) : single-pass summarization
# Long articles   (>  400 words) : chunk -> summarize each -> merge -> final pass

def summarize_article(
    text: str,
    chunk_max_words: int = 400,
    chunk_overlap: int = 50,
    verbose: bool = True
) -> str:

    cleaned = clean_text(text)
    word_count = len(cleaned.split())

    if word_count <= chunk_max_words:
        if verbose:
            print(f'Short article ({word_count} words) — single-pass summarization.')
        # For short articles, summarize_chunk uses its default max_new_tokens and min_new_tokens
        return summarize_chunk(cleaned)

    # Split into chunks
    chunks = split_into_chunks(cleaned, chunk_max_words, chunk_overlap)

    if verbose:
        print(f'Long article ({word_count} words) — split into {len(chunks)} chunks.')

    # Summarize each chunk
    partial_summaries = []
    for i, chunk in enumerate(chunks):
        if verbose:
            print(f'  Summarizing chunk {i + 1}/{len(chunks)}...')
        # Adjust max_new_tokens and min_new_tokens for partial summaries here
        partial = summarize_chunk(chunk, max_new_tokens=200, min_new_tokens=60)
        partial_summaries.append(partial)

    # Merge partial summaries and run a final summarization pass
    merged = ' '.join(partial_summaries)

    if verbose:
        print('  Running final summarization pass on merged chunks...')

    # Adjust max_new_tokens and min_new_tokens for the final summary here
    final_summary = summarize_chunk(
        merged,
        max_new_tokens=350,
        min_new_tokens=100
    )

    return final_summary

In [ ]:
# CELL 10
# Run summarization

summary = summarize_article(text, verbose=True)

print('\n' + '=' * 60)
print('TITLE:')
print(title)
print('\nSUMMARY:')
print(summary)
print('=' * 60)

In [ ]:
# CELL 11
# Summary statistics

original_word_count = len(text.split())
summary_word_count  = len(summary.split())
compression_ratio   = (
    (original_word_count - summary_word_count) / original_word_count
) * 100

print('STATISTICS')
print(f'Original words : {original_word_count}')
print(f'Summary words  : {summary_word_count}')
print(f'Compression    : {compression_ratio:.2f}%')

### Validation: ROUGE Scores

Untuk evaluasi abstractive summarizer, kita tetap menggunakan ROUGE sebagai proxy. Karena tidak ada reference summary buatan manusia, artikel asli digunakan sebagai referensi. Perlu diingat bahwa untuk abstractive summarization, skor ROUGE cenderung lebih rendah dibanding extractive — ini **normal**, karena model generate kata-kata baru, bukan menyalin kalimat.

In [ ]:
# CELL 12
# Validation: ROUGE Scores

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

scores = scorer.score(target=text, prediction=summary)

print('ROUGE Scores (vs. original article):')
print(f'{"Metric":<10} {"Precision":>10} {"Recall":>10} {"F1":>10}')
print('-' * 44)
for key, value in scores.items():
    print(
        f'{key:<10} '
        f'{value.precision:>10.4f} '
        f'{value.recall:>10.4f} '
        f'{value.fmeasure:>10.4f}'
    )